# Verification of the modulo-256 identities in Sage

Load the precomputed modular-symbol modules and the recorded intermediate elements in the presentations of linear relations. Check the defining equations on each cyclic generator, using Sage throughout.

The source presentations and Hecke matrices are taken as inputs. This notebook verifies the finite-source conditions; it does not reconstruct the Manin presentation or prove propagation to all degrees.

In [4]:
import gzip
import hashlib
import json
import sys
from pathlib import Path
from sage.all import *

ROOT = Path.cwd()
if not (ROOT / 'python').is_dir():
    raise FileNotFoundError('Open this notebook from the repository root.')
sys.path.insert(0, str(ROOT / 'python'))

from load_source_data import load_source_data
from verify_hecke_relations import relation_spec

SOURCE_DIRECTORY = ROOT / 'source_data/p2_mod256_native'
WITNESS_DIRECTORY = ROOT / 'verification_data/mod256_compact/T3_T5/packets'
if not WITNESS_DIRECTORY.is_dir():
    raise FileNotFoundError('The recorded verification-data directory is required.')


## 1. Define the relations

We use the notation of the manuscript's prime-2 section. Put $R_8=\mathbb{Z}/256\mathbb{Z}$ and $r=d\bmod128$.

The constants and polynomials are:

$$
c_{3,r}=1+3^{r+1},\qquad c_{5,r}=1+5^{r+1}\quad\text{in }R_8.
$$

$$
X_r=T_3-c_{3,r},\qquad F_r(Z)=Z^2-\varepsilon_r Z.
$$

Here $\varepsilon_r=1$ when $r\bmod16$ is $0,6,8,$ or $14$, and $\varepsilon_r=0$ otherwise.

### Conditions to verify

The source is the **unsigned** module $M=\mathbb{M}_d(R_8)$. On this module, define the division relation

$$
\mathcal{Z}_r=\mathfrak{D}_{X_r,\,2^7}(M).
$$

We check the following three conditions:

$$
\operatorname{dom}\mathcal{Z}_r=M,
$$

$$
F_r(\mathcal{Z}_r)\equiv0\pmod{2}\quad\text{on }M,
$$

$$
(T_5-c_{5,r})M=0.
$$

The division exponent is $\alpha_1=7$ and the terminal depth is $b=1$. The code variable `Z` stands for the source relation $\mathcal{Z}_r$, not the target endomorphism $Z_r$.

### Notation in the code

| Code | Manuscript |
| :--- | :--- |
| `c_3[r]` | $c_{3,r}$ |
| `c_5[r]` | $c_{5,r}$ |
| `varepsilon[r]` | $\varepsilon_r$ |
| `X[r]` | $X_r$ |
| `F[r]` | $F_r$ |

The identifiers `T3`, `T5`, and `Z` are retained in the archive format.

**Degrees checked:** the even induction-base degrees from 256 to 638, together with all lower even degrees from 0 to 254.


In [5]:
p = 2
m = 8
R = Integers(p^m)
S.<T3,T5,Z> = PolynomialRing(R)
period = euler_phi(p^m)
a_m = p^m * (p-1)             # a_8 = 256
b_m = p^(m-1) * (p+1)         # b_8 = 384
degrees = tuple(range(0, a_m+b_m, 2))

# Manuscript notation: c_{3,r}, c_{5,r}, epsilon_r, X_r, F_r.
alpha_1 = 7
b = 1
c_3, c_5, varepsilon, X, F = {}, {}, {}, {}, {}
specifications = {}
for r in range(0, period, 2):
    c_3[r] = R(1 + 3^(r+1))
    c_5[r] = R(1 + 5^(r+1))
    varepsilon[r] = 1 if r % 16 in (0,6,8,14) else 0
    X[r] = T3-c_3[r]
    F[r] = Z^2-varepsilon[r]*Z
    specifications[r] = relation_spec(
        [('T5', T5-c_5[r], m),
         ('T3_full_domain', Z, 0),
         ('T3_terminal', F[r], b)],
        hecke_operators={'T3': 3, 'T5': 5},
        divisions={'Z': (X[r], alpha_1)},
        witness_semantics='independent_monomials',
        max_howell_dimension=4096,
    )

# Check that these explicit definitions match the saved packets' specification.
plan = json.loads((ROOT / 'relations/p2_mod256_T3_T5_native.json').read_text())
assert all(specifications[r] == plan['relation_specifications'][str(r)] for r in specifications)
print('Working modulus:', p^m)
print('Degrees:', degrees)


Working modulus: 256
Degrees: (0, 2, 4, 6, 8, 10, 12, 14, 16, 18, 20, 22, 24, 26, 28, 30, 32, 34, 36, 38, 40, 42, 44, 46, 48, 50, 52, 54, 56, 58, 60, 62, 64, 66, 68, 70, 72, 74, 76, 78, 80, 82, 84, 86, 88, 90, 92, 94, 96, 98, 100, 102, 104, 106, 108, 110, 112, 114, 116, 118, 120, 122, 124, 126, 128, 130, 132, 134, 136, 138, 140, 142, 144, 146, 148, 150, 152, 154, 156, 158, 160, 162, 164, 166, 168, 170, 172, 174, 176, 178, 180, 182, 184, 186, 188, 190, 192, 194, 196, 198, 200, 202, 204, 206, 208, 210, 212, 214, 216, 218, 220, 222, 224, 226, 228, 230, 232, 234, 236, 238, 240, 242, 244, 246, 248, 250, 252, 254, 256, 258, 260, 262, 264, 266, 268, 270, 272, 274, 276, 278, 280, 282, 284, 286, 288, 290, 292, 294, 296, 298, 300, 302, 304, 306, 308, 310, 312, 314, 316, 318, 320, 322, 324, 326, 328, 330, 332, 334, 336, 338, 340, 342, 344, 346, 348, 350, 352, 354, 356, 358, 360, 362, 364, 366, 368, 370, 372, 374, 376, 378, 380, 382, 384, 386, 388, 390, 392, 394, 396, 398, 400, 402, 404, 406, 408,

## 2. Verify the defining equations

For every cyclic generator $x_0$ of $M=\mathbb{M}_d(R_8)$, recover the recorded intermediate elements $x_1,x_2,\rho\in M$ and check:

$$
\begin{aligned}
X_r x_0 &= 128x_1,\\
X_r x_1 &= 128x_2,\\
x_2-\varepsilon_r x_1 &= 2\rho,\\
(T_5-c_{5,r})x_0 &= 0.
\end{aligned}
$$

The first two equations give a common chain

$$
x_0\mathcal{Z}_r x_1\mathcal{Z}_r x_2.
$$

The third equation places its polynomial output in $2M$. This supplies an element of the expanded polynomial relation, as in the manuscript. It does **not** assert that division defines an endomorphism of the source.

The presented relations are linear, so checking all cyclic generators suffices for the whole module. Every coordinate is checked modulo its own cyclic order.

### Reading the recorded data

The saved data specify coordinatewise preimages with zero kernel corrections. This notebook supports that recorded construction only.

The archive names `WITNESS_DIRECTORY` and `compact-relation-witness` refer to these recorded intermediate elements; they introduce no additional mathematical notion.

The file identifier (called the **binding**) checks that the recorded data belong to the chosen source and relations. The separate equation checks establish the finite-source conditions.

For a quick trial, set `degrees = (0, 12, 256)` before running the last cell.


In [6]:
def compact_json(value):
    """
    Serialize the source data and presentation in the archive's JSON format.
    Return the text used to identify the corresponding verification data.
    Values must already be JSON-compatible, including Python integers.
    """
    return json.dumps(value, separators=(',', ':'), ensure_ascii=False)

def nim_sequence(values):
    """
    Encode a sequence of integers in the archive's '@[1, 2, ...]' format.
    This is used only for the identifier of the recorded data; no Nim
    computation is performed.
    """
    # Text encoding used by the producer's packet binding.
    return '@[' + ', '.join(str(int(v)) for v in values) + ']'

def packet_binding(spec, metadata, moduli, operators):
    """
    Identify the recorded data for this source and these presentations.
    
    The archive calls this identifier a 'binding'. It is computed from
    the coefficient degree, modulus, source metadata, cyclic coordinates,
    Hecke matrices, tested generators, and the specified relations.
    
    Changing the degree, coordinates, or presentation generally requires
    different intermediate elements. Reproducing the stored identifier
    ensures that the data correspond to the inputs under consideration.
    
    This file-consistency check does not establish a mathematical identity.
    The defining equations of the presentations are checked separately
    by replay_defined_relations.
    """
    rank = len(moduli)
    digest = hashlib.sha1()
    parts = [
        compact_json(spec), f'{p}:{m}', compact_json(metadata),
        nim_sequence(moduli),
        f'{rank}:{rank}:' + nim_sequence(
            1 if i == j else 0 for i in range(rank) for j in range(rank)
        ),
    ]
    for name in spec['variables']:
        if name in operators:
            parts.append(name + ':' + nim_sequence(operators[name].list()))
    for part in parts:
        digest.update(part.encode())
    return digest.hexdigest().upper()

def load_canonical_packet(binding, name, metadata, rank):
    """
    Load the recorded intermediate-element data for one relation.
    
    Check the source identifier, relation name, working modulus, and
    number of cyclic generators. Return the decoded archive record.
    
    The saved data specify how to choose the intermediate elements
    x_1, x_2, ... in the defining equations. For p^alpha*y = u,
    reduce each coordinate of u to its least nonnegative representative
    and divide by p^alpha, provided the required divisibility holds.
    The resulting equation is checked separately in the source module.
    
    The present files use these choices without further adjustments.
    If a file specifies different choices, this verifier stops because
    it does not implement their reconstruction. No search for
    intermediate elements is performed. These elementwise choices
    need not define an endomorphism of the source.
    """
    suffix = hashlib.sha1(name.encode()).hexdigest().upper()
    path = WITNESS_DIRECTORY / f'{binding}_{suffix}.json.gz'
    with gzip.open(path, 'rt') as stream:
        packet = json.load(stream)
    assert packet['schema'] in (
        'hecke.compact-relation-witness.v1',
        'hecke.compact-relation-witness.v2',
    ), path
    assert packet['binding'] == binding and packet['relation'] == name, path
    assert packet['source'] == metadata, path
    assert packet['working_modulus'] == p^m, path
    assert packet['input_count'] == rank, path
    # Only the recorded common chains with zero kernel corrections
    # are supported here; other presentations of elements are rejected.
    if packet['independent'] is not False or packet['choices'] != [] or 'recipe' in packet:
        raise NotImplementedError(f'{path}: unsupported recorded intermediate-element construction')
    return packet

def mixed_zero(value, moduli):
    """
    Check equality to zero in the source module in cyclic coordinates.
    
    The rows represent elements of M = direct_sum_j Z/moduli[j]Z.
    Each column is reduced modulo its own cyclic order, rather than the
    ambient modulus. Return True precisely when all rows represent zero.
    For the zero module, this condition holds vacuously.
    """
    return all(value[i,j] % order == 0
               for i in range(value.nrows())
               for j, order in enumerate(moduli))

def canonical_preimage(rhs, divisor, moduli):
    """
    Recover the recorded y in the relation rhs = divisor*y on M.
    
    The rows of rhs are elements of M in cyclic coordinates. The divisor
    and cyclic orders are powers of the same prime. In each coordinate,
    take the least nonnegative residue of rhs and divide it by divisor,
    using the coordinatewise choice specified by the recorded data.
    
    Return the rows y after checking divisor*y = rhs in M. Failure of
    the required divisibility raises ArithmeticError. These are choices
    of intermediate elements, not a globally defined divided endomorphism.
    """
    # Each row is an element, not necessarily the image of an endomorphism.
    output = zero_matrix(ZZ, rhs.nrows(), rhs.ncols())
    for i in range(rhs.nrows()):
        for j, order in enumerate(moduli):
            entry = ZZ(rhs[i,j]) % order
            if entry % gcd(divisor, order):
                raise ArithmeticError(f'No recorded preimage: row {i}, coordinate {j}')
            output[i,j] = entry // divisor
    assert mixed_zero(divisor * output - rhs, moduli)
    return output

def ordinary_polynomial_matrix(terms, variables, operators, rank):
    """
    Evaluate an ordinary polynomial in the archived Hecke operators.
    
    The sparse terms are pairs [coefficient, exponent list], with
    exponents ordered by variables. Operators act on row vectors.
    A constant acts as the corresponding scalar identity matrix.
    Return an integer matrix representing the resulting action;
    equality in M is checked later using its cyclic coordinate orders.
    
    Only ordinary operators are accepted by this helper. Polynomial
    evaluation at a division relation is handled separately.
    """
    value = zero_matrix(ZZ, rank)
    for coefficient, powers in terms:
        term = identity_matrix(ZZ, rank)
        for name, exponent in reversed(list(zip(variables, powers))):
            if exponent:
                if name not in operators:
                    raise ValueError('Expected an ordinary polynomial')
                term = term * operators[name]^exponent
        value += ZZ(coefficient) * term
    return value

def replay_defined_relations(spec, operators, moduli):
    """
    Verify the specified ordinary and presented-relation conditions on M.
    
    For the single division relation in spec, recover a common chain
    x_0, ..., x_n for each cyclic generator x_0, and check every equation
    defining that chain. For F(Z) = sum_j c_j Z^j, the element
    y = sum_j c_j x_j then belongs to F(cal_Z)(x_0), by the manuscript's
    definition of polynomial evaluation at linear relations.
    
    Check y = p^b*rho with rho in M. This establishes that
    F(cal_Z)(x_0) intersects p^b M. Checking all cyclic generators
    suffices, since the corresponding presented relation is linear.
    Ordinary relations are checked using the same membership condition.
    
    The recorded common-chain construction must have been validated by
    load_canonical_packet. Return None on success. This helper verifies
    the finite-source conditions only; it does not establish propagation
    or define the divided operator on the free target lattice.
    """
    rank = len(moduli)
    identity = identity_matrix(ZZ, rank)
    assert len(spec['divisions']) == 1
    division = spec['divisions'][0]
    z_index = spec['variables'].index(division['variable'])
    numerator_action = ordinary_polynomial_matrix(
        division['numerator'], spec['variables'], operators, rank
    )
    divisor = p^division['power']

    for relation in spec['relations']:
        terms = relation['polynomial']
        divided = any(powers[z_index] for coefficient, powers in terms)
        if not divided:
            terminal = ordinary_polynomial_matrix(
                terms, spec['variables'], operators, rank
            )
        else:
            # This simple replay supports terminal polynomials in Z alone.
            assert all(all(e == 0 for j, e in enumerate(powers) if j != z_index)
                       for coefficient, powers in terms)
            degree = max(powers[z_index] for coefficient, powers in terms)
            witnesses = [identity]
            for j in range(degree):
                rhs = witnesses[-1] * numerator_action
                witness = canonical_preimage(rhs, divisor, moduli)
                assert mixed_zero(rhs - divisor*witness, moduli)
                witnesses.append(witness)
            terminal = zero_matrix(ZZ, rank)
            for coefficient, powers in terms:
                terminal += ZZ(coefficient) * witnesses[powers[z_index]]

        terminal_divisor = p^relation['terminal_power']
        rho = canonical_preimage(terminal, terminal_divisor, moduli)
        assert mixed_zero(terminal - terminal_divisor*rho, moduli)

def replay_mod256_degree(d):
    """
    Verify the source identities in the even coefficient degree d.
    
    Load the unsigned modular-symbol module M_d(R_8) in cyclic coordinates
    and the archived Hecke matrices. Check compatibility with the cyclic
    orders and identify the recorded data for the specified presentations.
    Then verify their defining equations and congruence conditions in Sage.
    
    Return the degree, number of cyclic coordinates, number of archive
    records read, and success. The Manin presentation and Hecke actions
    are inputs: they are not reconstructed or checked for descent here.
    """
    data = load_source_data(R, d, 0, SOURCE_DIRECTORY / f'degree_{d}.npz')
    spec = specifications[d % period]
    moduli = [int(v) for v in data['coordinates']['coordinate_moduli']]
    rank = len(moduli)
    assert all(order > 1 and (p^m) % order == 0 for order in moduli)
    operators = {
        name: matrix(ZZ, data['archived_hecke_matrices'][n]['normalized_matrix'])
        for name, n in spec['hecke_operators'].items()
    }
    for action in operators.values():
        assert action.dimensions() == (rank, rank)
        assert all(moduli[i] * action[i,j] % moduli[j] == 0
                   for i in range(rank) for j in range(rank))

    metadata = dict(data['source_metadata'])
    metadata.update(degree=int(d), orientation=int(0), sign=int(0),
                    source_path=data['source_path'], source_sha256=data['source_sha256'])
    # Identify the recorded intermediate-element data for this source
    # and these presentations before checking their defining equations.
    binding = packet_binding(spec, metadata, moduli, operators)
    for relation in spec['relations']:
        load_canonical_packet(binding, relation['name'], metadata, rank)

    # Now do the mathematical check: substitute the recorded choices
    # and verify every division and terminal equation in the module.
    replay_defined_relations(spec, operators, moduli)
    return {'degree': d, 'rank': rank, 'packets_loaded': len(spec['relations']), 'passed': True}

results = []
for d in degrees:
    test = replay_mod256_degree(d)
    results.append(test)
    print(test, flush=True)

print('Degrees replayed:', len(results))
print('ALL REQUESTED MODULO-256 SOURCE IDENTITIES REPLAYED IN SAGE')


{'degree': 0, 'rank': 0, 'packets_loaded': 3, 'passed': True}


{'degree': 2, 'rank': 1, 'packets_loaded': 3, 'passed': True}
{'degree': 4, 'rank': 2, 'packets_loaded': 3, 'passed': True}
{'degree': 6, 'rank': 2, 'packets_loaded': 3, 'passed': True}
{'degree': 8, 'rank': 3, 'packets_loaded': 3, 'passed': True}
{'degree': 10, 'rank': 4, 'packets_loaded': 3, 'passed': True}
{'degree': 12, 'rank': 4, 'packets_loaded': 3, 'passed': True}
{'degree': 14, 'rank': 5, 'packets_loaded': 3, 'passed': True}
{'degree': 16, 'rank': 6, 'packets_loaded': 3, 'passed': True}
{'degree': 18, 'rank': 6, 'packets_loaded': 3, 'passed': True}
{'degree': 20, 'rank': 7, 'packets_loaded': 3, 'passed': True}
{'degree': 22, 'rank': 8, 'packets_loaded': 3, 'passed': True}
{'degree': 24, 'rank': 8, 'packets_loaded': 3, 'passed': True}
{'degree': 26, 'rank': 9, 'packets_loaded': 3, 'passed': True}
{'degree': 28, 'rank': 10, 'packets_loaded': 3, 'passed': True}
{'degree': 30, 'rank': 10, 'packets_loaded': 3, 'passed': True}
{'degree': 32, 'rank': 11, 'packets_loaded': 3, 'passed':

## Strong realization of the permitted signatures

Check that every permitted signature is realized by a normalized
cuspidal level-one eigenform. Use the constants
$c_{3,r},c_{5,r}$ and $\varepsilon_r$ defined above,
with $r=k-2\bmod128$. The permitted pairs are
$(c_{3,r},c_{5,r})$, and also $(c_{3,r}+128,c_{5,r})$ when
$\varepsilon_r=1$. Group them by **weight** residue $k\bmod64$.

Compute normalized cuspidal level-one eigenforms through weight 90,
including every orbit and every prime above 2. At a prime of
ramification index $e$, the KRW reduction modulo 256 is reduction
modulo the prime ideal to exponent $7e+1$. An integer representative
is accepted only after testing this congruence; it is not assumed.

Run the imports and relation definitions first. These cells do not
rerun the source verification. Unlike the saved-data checks in the
other playgrounds, this section computes eigenforms and may take time.
Matching all 48 signatures proves the finite realization step.
The all-weight conclusion also uses the source identities,
propagation and completed-Hecke generation from the manuscript.


In [3]:
STRONG_WEIGHT_BOUND_MOD256 = 90
STRONG_WEIGHT_PERIOD_MOD256 = 64

# Use the relations already displayed above; k is the classical weight.
strong_mod256_expected = {}
for k_residue in range(0, STRONG_WEIGHT_PERIOD_MOD256, 2):
    degree_residue = (k_residue - 2) % 128
    pairs = {(ZZ(c_3[degree_residue]), ZZ(c_5[degree_residue]))}
    if varepsilon[degree_residue] == 1:
        pairs.add((ZZ((c_3[degree_residue] + 128)), ZZ(c_5[degree_residue])))
    strong_mod256_expected[k_residue] = pairs

assert sum(map(len, strong_mod256_expected.values())) == 48


NameError: name 'c_3' is not defined

In [1]:
def mod256_krw_integer(x, nf, prime_ideal):
    """
    Find the integer representative of x in the valuative quotient.
    For ramification index e, test membership in the prime ideal
    to exponent 7e+1. Reject a residue which is not represented by
    exactly one integer in 0,...,255.
    """
    e = prime_ideal[2]
    exponent = e*(8 - 1) + 1
    candidates = [
        value for value in range(256)
        if nf.idealval(x - value, prime_ideal) >= exponent
    ]
    if len(candidates) != 1:
        raise ValueError(f'Expected one integer KRW residue, found {candidates}')
    return ZZ(candidates[0])


def compute_mod256_strong_signatures(bound):
    """
    Compute (k,a_3,a_5) modulo 256 for strong eigenforms of weight at most bound.
    Include every normalized cuspidal level-one eigenform orbit.
    For nonrational coefficient fields, use a 2-maximal order and
    test the KRW reduction at every prime above 2.
    """
    residues = Integers(256)
    signatures = []
    for weight in range(12, bound + 1, 2):
        cusp_forms = CuspForms(1, weight)
        if cusp_forms.dimension() == 0:
            continue
        print(f'Computing strong signatures at weight {weight}', flush=True)
        for eigenform in cusp_forms.newforms(names='a'):
            coefficient_field = eigenform.base_ring()
            if coefficient_field == QQ:
                signatures.append((weight, ZZ(residues(eigenform[3])),
                                   ZZ(residues(eigenform[5]))))
                continue
            polynomial = coefficient_field.pari_polynomial('y')
            nf = pari([polynomial, [2]]).nfinit(4)
            a3, a5 = pari(eigenform[3]), pari(eigenform[5])
            for prime_ideal in nf.idealprimedec(2):
                signatures.append((
                    weight,
                    mod256_krw_integer(a3, nf, prime_ideal),
                    mod256_krw_integer(a5, nf, prime_ideal),
                ))
    return signatures


In [ ]:
strong_mod256_signatures = compute_mod256_strong_signatures(STRONG_WEIGHT_BOUND_MOD256)
strong_mod256_realized = {}
for weight, a3, a5 in strong_mod256_signatures:
    k_residue = weight % STRONG_WEIGHT_PERIOD_MOD256
    strong_mod256_realized.setdefault(k_residue, set()).add((a3, a5))


In [ ]:
print('k mod 64 | (a_3, a_5) mod 256')
print('-' * 100)
for k_residue in sorted(strong_mod256_expected):
    pairs = ', '.join(
        f'({a1}, {a2})'
        for a1, a2 in sorted(strong_mod256_realized.get(k_residue, set()))
    )
    print(f'{k_residue:3d} | {pairs}')

assert strong_mod256_realized == strong_mod256_expected
print('Every permitted modulo-256 signature has a strong representative.')
